# Leviathan Threat Classifier — 2D CNN

### Main Features
- Sequence-to-image transformation
- Multi-channel engineered features
- 2D CNN architecture
- Stratified K-Fold Cross Validation
- Learning rate scheduling
- Early stopping
- Final submission generation

### Submitted by:
    Saksham Goyal
    25EE10088


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully")

Libraries imported successfully


## 1. Load Dataset

The dataset contains:
- `sequence_id` : Unique sequence identifier
- `timestep` : Time index
- `sensor_0` to `sensor_24` : Sensor readings
- `level` : Target threat class


In [2]:
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

SENSOR_COLS = [f"sensor_{i}" for i in range(25)]

N_TIMESTEPS = 10
N_SENSORS   = 25

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (300000, 28)
Test shape : (100000, 27)


## 2. Tensor Construction

Each sequence is reshaped into:

`(timesteps, sensors)` → `(10, 25)`

This allows the model to treat the data like a small image.


In [3]:
def build_tensor(df, sensor_cols, n_timesteps=10):
    df = df.sort_values(["sequence_id", "timestep"])
    vals = df[sensor_cols].values.reshape(-1, n_timesteps, len(sensor_cols))
    return vals

X_raw = build_tensor(train_df, SENSOR_COLS)
X_test = build_tensor(test_df, SENSOR_COLS)

y_all = train_df.groupby("sequence_id")["level"].first().values.astype(int)

print("Train tensor shape:", X_raw.shape)
print("Test tensor shape :", X_test.shape)

Train tensor shape: (30000, 10, 25)
Test tensor shape : (10000, 10, 25)


## 3. Feature Scaling

Standardization is applied sensor-wise using `StandardScaler`.

This improves optimization stability during neural network training.


In [4]:
scaler = StandardScaler()

X_flat = X_raw.reshape(-1, N_SENSORS)
scaler.fit(X_flat)

def scale(X, scaler):
    n = X.shape[0]
    return scaler.transform(
        X.reshape(-1, N_SENSORS)
    ).reshape(n, N_TIMESTEPS, N_SENSORS)

X_scaled = scale(X_raw, scaler)
X_test_scaled = scale(X_test, scaler)

print("Scaling completed")

Scaling completed


## 4. Feature Engineering

Three channels are created:

1. Original normalized signal
2. Temporal difference channel
3. Z-score normalized channel

Final shape:

`(N, 10, 25, 3)`


In [5]:
X_img = X_scaled[..., np.newaxis]
X_test_img = X_test_scaled[..., np.newaxis]

def make_diff_channel(X):
    d = np.diff(X, axis=1, prepend=X[:, :1, :])
    return d[..., np.newaxis]

def make_zscore_channel(X):
    mu  = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True) + 1e-6
    z   = (X - mu) / std
    return z[..., np.newaxis]

X_diff = make_diff_channel(X_scaled)
X_zscore = make_zscore_channel(X_scaled)

X_multi = np.concatenate(
    [X_img, X_diff, X_zscore],
    axis=-1
)

X_test_diff = make_diff_channel(X_test_scaled)
X_test_zscore = make_zscore_channel(X_test_scaled)

X_test_multi = np.concatenate(
    [X_test_img, X_test_diff, X_test_zscore],
    axis=-1
)

print("Final training tensor:", X_multi.shape)

Final training tensor: (30000, 10, 25, 3)


## 5. Model Architecture

The architecture contains:

- Conv2D layers
- Batch Normalization
- MaxPooling
- Dropout Regularization
- Global Average Pooling
- Dense Classification Head

Padding is used with:

```python
padding="same"
```

This preserves spatial dimensions after convolution.


In [6]:
def build_model(input_shape=(10, 25, 3), n_classes=4):

    inp = keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(
        64, (3, 5),
        padding="same",
        activation="relu"
    )(inp)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        64, (3, 5),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((1, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(
        128, (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        128, (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(
        256, (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256, (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    # Dense Head
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inp, out)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 10, 25, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 10, 25, 64)          │           2,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 10, 25, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 10, 25, 64)          │          61,504 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 10, 25, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 10, 12, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 10, 12, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 10, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 10, 12, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 10, 12, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 10, 12, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 5, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 5, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 5, 6, 256)           │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 5, 6, 256)           │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 5, 6, 256)           │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 5, 6, 256)           │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │              

 Total params: 1,274,948 (4.86 MB)

 Trainable params: 1,272,644 (4.85 MB)

 Non-trainable params: 2,304 (9.00 KB)

## 6. Cross Validation Training

Stratified K-Fold validation is used to:

- improve generalization
- reduce overfitting
- obtain robust validation accuracy

Training callbacks:
- ReduceLROnPlateau
- EarlyStopping


In [7]:
N_FOLDS = 3
EPOCHS = 45
BATCH = 128

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros((len(y_all), 4))
test_preds = np.zeros((X_test_multi.shape[0], 4))

fold_accs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_multi, y_all)):

    print(f"\nFold {fold+1}/{N_FOLDS}")

    X_tr, X_va = X_multi[tr_idx], X_multi[va_idx]
    y_tr, y_va = y_all[tr_idx], y_all[va_idx]

    model = build_model()

    cb = [
        callbacks.ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        ),

        callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=15,
            restore_best_weights=True
        )
    ]

    model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=cb,
        verbose=1
    )

    va_prob = model.predict(X_va, verbose=0)
    va_pred = va_prob.argmax(axis=1)

    acc = accuracy_score(y_va, va_pred)

    fold_accs.append(acc)

    print("Fold Accuracy:", acc)

    oof_preds[va_idx] += va_prob

    test_preds += model.predict(
        X_test_multi,
        verbose=0
    )

    keras.backend.clear_session()


Fold 1/3
Epoch 1/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 96s 563ms/step - accuracy: 0.2600 - loss: 1.6733 - val_accuracy: 0.2648 - val_loss: 1.4101 - learning_rate: 0.0010
Epoch 2/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 88s 559ms/step - accuracy: 0.2699 - loss: 1.4614 - val_accuracy: 0.2032 - val_loss: 1.5873 - learning_rate: 0.0010
Epoch 3/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 88s 561ms/step - accuracy: 0.2736 - loss: 1.4161 - val_accuracy: 0.2207 - val_loss: 1.4248 - learning_rate: 0.0010
Epoch 4/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 87s 554ms/step - accuracy: 0.2812 - loss: 1.3960 - val_accuracy: 0.2842 - val_loss: 1.3807 - learning_rate: 0.0010
Epoch 5/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 87s 552ms/step - accuracy: 0.2919 - loss: 1.3863 - val_accuracy: 0.2746 - val_loss: 1.3798 - learning_rate: 0.0010
Epoch 6/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 86s 546ms/step - accuracy: 0.2910 - loss: 1.3838 - val_accuracy: 0.3034 - val_loss: 1.3756 - learning_rate: 0.0010
Epoch 7/45
157/157 ━━━━━━━━━━━━━━━━━━━━ 87s 554ms/step - acc

## 7. Final Evaluation

The Out-of-Fold (OOF) score is computed after combining predictions from all folds.


In [8]:
oof_final = oof_preds.argmax(axis=1)

oof_acc = accuracy_score(y_all, oof_final)

print("OOF Accuracy:", oof_acc)
print("Per Fold Accuracies:", fold_accs)

OOF Accuracy: 0.8119666666666666
Per Fold Accuracies: [0.8135, 0.8147, 0.8077]


## 8. Submission File Generation

Predictions are averaged across folds and converted into final labels.


In [9]:
test_preds /= N_FOLDS

final_preds = test_preds.argmax(axis=1)

test_seq_ids = (
    test_df.sort_values(["sequence_id", "timestep"])
           .groupby("sequence_id")["sequence_id"]
           .first()
           .index
           .values
)

sub = pd.DataFrame({
    "sequence_id": test_seq_ids,
    "level": final_preds
})

out_path = "submission2d.csv"

sub.to_csv(out_path, index=False)

print("Submission saved:", out_path)
print(sub.head())

Submission saved: submission2d.csv
   sequence_id  level
0        30000      3
1        30001      2
2        30002      3
3        30003      3
4        30004      3


# Conclusion

This solution uses a multi-channel 2D CNN for sequence classification.

### Key strengths:
- Temporal + statistical feature engineering
- CNN-based spatial learning
- Cross-validation robustness
- Regularization through dropout and batch normalization

### Final Results

- Best Fold Accuracy: ~82%
- OOF Accuracy: ~81%
- Architecture Used: Multi-channel 2D CNN
- Validation Strategy: Stratified 3-Fold Cross Validation
